## IBM Quantum API Setup & Hardware Execution Notebook



---  
##  PART 1: How to Get & Setup Your IBM Quantum API Key

### **Step 1: Obtain Your IBM Quantum API Key**
1. Go to the **[IBM Quantum Platform](https://quantum.cloud.ibm.com/)** and sign in or create a free account.
2. On your main dashboard, locate the **API Key / API Token** section.
3. Click **Create API Key** (or copy your existing 44-character API key).
4. **Copy and safely store your key** — you won't be able to view the raw key again after leaving the page.

---  
### **Step 2: Choose How to Authenticate**

#### **Option A: Save Locally on Personal Computer (Recommended)**
Saves your key securely to disk (`$HOME/.qiskit/qiskit-ibm.json`) so Qiskit automatically authenticates in future sessions:
```python
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="YOUR_44_CHARACTER_API_KEY",
    overwrite=True
)
```

#### **Option B: Prompt Interactively (Safest for Shared Notebooks)**
Prompts you for your key securely at runtime using Python's `getpass` module without storing raw credentials inside `.ipynb` text files:
```python
import getpass
from qiskit_ibm_runtime import QiskitRuntimeService

api_key = getpass.getpass(prompt="Enter IBM Quantum API Key: ")
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=api_key)
```

---  
##  PART 2: Package Installation & Verification

In [ ]:
!pip install qiskit qiskit-ibm-runtime matplotlib scipy

In [ ]:
import qiskit
import qiskit_ibm_runtime

print(f"Qiskit Core Version: {qiskit.__version__}")
print(f"IBM Runtime Service Version: {qiskit_ibm_runtime.__version__}")

---  
##  PART 3: Application — Variational Quantum Eigensolver (VQE)
*(Syllabus Unit V — Applications of Quantum Computing)*

We construct a Variational Quantum Eigensolver (VQE) to search for the ground state energy of a quantum Hamiltonian operator $H = 0.5 Z_0 - 1.0 X_0$.

In [ ]:
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector

# 1. Define Hamiltonian Operator H = 0.5*Z - 1.0*X
H = SparsePauliOp.from_list([("Z", 0.5), ("X", -1.0)])

# 2. Parameterized Ansatz State Circuit
def build_ansatz(theta):
    qc = QuantumCircuit(1)
    qc.ry(theta[0], 0)
    qc.rz(theta[1], 0)
    return qc

# 3. Local Expectation Value Evaluator
def energy_objective(theta):
    psi = Statevector.from_instruction(build_ansatz(theta))
    return float(np.real(psi.expectation_value(H)))

# 4. Classical Optimization (COBYLA)
initial_params = [0.1, 0.1]
opt_res = minimize(energy_objective, initial_params, method="COBYLA")

exact_energy = np.linalg.eigvalsh(H.to_matrix())[0]
print(f"Optimized VQE Calculated Energy: {opt_res.fun:.5f}")
print(f"Exact Analytical Ground Energy:  {exact_energy:.5f}")

---  
##  PART 4: Authenticate with IBM Quantum Platform

In [ ]:
import getpass
from qiskit_ibm_runtime import QiskitRuntimeService

# Enter your 44-character API key securely
api_key = getpass.getpass(prompt="Paste your IBM Quantum API Key: ")

# Initialize Runtime Service
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=api_key)
print(" Successfully connected to IBM Quantum Account:::", service.active_account())

---  
##  PART 5: Query Cloud Hardware & Select Backend

In [ ]:
# Filter for real, online IBM Quantum QPUs
backends = service.backends(operational=True, simulator=False)
print("Available Real Quantum Systems:")
for b in backends:
    print(f" • {b.name}: {b.num_qubits} Qubits | Pending Jobs: {b.status().pending_jobs}")

# Select the least busy backend
target_backend = service.least_busy(operational=True, simulator=False)
print(f"\n Selected Target Backend: {target_backend.name}")

---  
##  PART 6: Transpile & Run Job on IBM Quantum Hardware

In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

# 1. Prepare Optimal State Circuit with Measurement
opt_circuit = build_ansatz(opt_res.x)
opt_circuit.measure_all()

# 2. Transpile for Hardware ISA (Instruction Set Architecture)
pm = generate_preset_pass_manager(target=target_backend.target, optimization_level=1)
isa_circuit = pm.run(opt_circuit)

print(f"Original Circuit Depth:   {opt_circuit.depth()}")
print(f"Hardware Transpiled Depth: {isa_circuit.depth()}")

# 3. Execute Job via SamplerV2 Primitive
sampler = Sampler(mode=target_backend)
print(f"Submitting job to {target_backend.name}...")
job = sampler.run([isa_circuit], shots=1000)
print(f"Job Submitted Successfully! Job ID: {job.job_id()}")

# 4. Fetch Hardware Results
result = job.result()
pub_result = result[0]
counts = pub_result.data.meas.get_counts()

print("\nExecution Counts from Real Quantum Hardware:", counts)
plot_histogram(counts)
plt.show()

## Reference

https://quantum.cloud.ibm.com/docs/en/tutorials

https://www.youtube.com/qiskit

https://quantum.cloud.ibm.com/docs/en/guides
